# 6. Hypothesis Testing — Validating a Claim Before Trusting It

**Building a Heart Disease Risk-Screening System — Notebook 6 of 12, Stage 4: Validating Claims Before Trusting Them**

Notebook 5 found that disease prevalence looks different between male and female
patients in this registry. **Is that a real difference, or could a registry this
size show a gap like that by chance alone?** This notebook builds the general
hypothesis-testing framework used to answer questions like this rigorously — and
sets up the decision table Notebooks 7 and 8 use to pick the *specific* test that
fits a given question.

## The topic

Hypothesis testing is a structured way to ask: "if there were truly no effect,
how surprising would data like this be?" It doesn't prove an effect is real — it
quantifies how consistent the data is with there being no effect at all.

## Why it matters for this system

Before any relationship found in Notebook 5 gets built into the model in Notebook
9, it needs to survive this check. Skip it, and the system risks encoding
noise — patterns in this specific 438-patient registry that wouldn't replicate in
the next batch of patients — as if they were real clinical signal.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
male_rate = df.loc[df["sex"] == 1, "target"].mean()
female_rate = df.loc[df["sex"] == 0, "target"].mean()
print(f"Disease rate, male patients:   {male_rate:.1%}  (n={sum(df['sex']==1)})")
print(f"Disease rate, female patients: {female_rate:.1%}  (n={sum(df['sex']==0)})")

## The toolkit — the general framework

| Step | What it does |
|---|---|
| **State $H_0$ / $H_1$** | Fix, in advance, what "no effect" and "an effect" mean |
| **Choose a test statistic** | A number that summarizes the data relevant to the hypothesis |
| **Compute a p-value** | $P(\text{data this extreme or more} \mid H_0 \text{ true})$ |
| **Compare to $\alpha$** | The pre-committed significance threshold (commonly 0.05) |
| **Check power / effect size** | Was the sample even big enough to detect a real effect, and does the effect *matter*, not just exist |

## How to choose *which specific test* — a preview of Notebooks 7-8

| Question shape | Test |
|---|---|
| Comparing a *proportion* between two groups | Two-proportion z-test (this notebook) |
| Comparing a *continuous mean* between two groups | t-test (Notebook 7) |
| Comparing *categorical distributions* across groups | Chi-square test (Notebook 8) |

The question's shape — proportion, mean, or categorical distribution — determines
which specific test applies; the surrounding logic (hypotheses, p-value, $\alpha$,
power) is identical across all three, which is exactly why it's worth building once
here before specializing.

## Applied to the registry

### State the hypothesis before looking at the test result

$H_0$: male and female patients have the *same* true disease rate.
$H_1$: they differ. This was effectively already fixed by the question Notebook 5
raised — the discipline is writing it down before running the test, not after.

### The p-value, computed correctly for a proportion comparison

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

count = np.array([df.loc[df["sex"]==1, "target"].sum(), df.loc[df["sex"]==0, "target"].sum()])
nobs = np.array([sum(df["sex"]==1), sum(df["sex"]==0)])

z_stat, p_value = proportions_ztest(count, nobs)
print(f"z-statistic: {z_stat:.3f}")
print(f"p-value: {p_value:.2e}")

### Reading the result against a pre-chosen $\alpha$

In [ ]:
alpha = 0.05
observed_gap = male_rate - female_rate

if p_value < alpha:
    print(f"p={p_value:.2e} < alpha={alpha} -> reject H0.")
    print(f"The {observed_gap:.1%} gap in disease rate is statistically significant.")
else:
    print(f"p={p_value:.2e} >= alpha={alpha} -> fail to reject H0.")

### Type I and Type II errors — the two ways this can go wrong

**Type I** (false positive): the system treats sex as a real risk-differentiator
when it isn't — every downstream use of that "finding" is built on noise. **Type
II** (false negative): the registry is too small to detect a real difference that
exists — the system misses a genuine risk factor. $\alpha$ controls the Type I
rate directly; **power** measures the ability to avoid a Type II error.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

effect_size = proportion_effectsize(male_rate, female_rate)
achieved_power = NormalIndPower().power(effect_size, nobs1=nobs[0], alpha=alpha, ratio=nobs[1]/nobs[0])
print(f"Effect size (Cohen's h): {effect_size:.3f}")
print(f"Achieved power at this registry's size: {achieved_power:.1%}")
print("Power well below 80% would mean a real difference this size has a meaningful chance")
print("of being missed entirely -- worth knowing before concluding 'no significant difference'")
print("means 'no real difference'.")

### Effect size: statistically significant isn't automatically important

A large registry could make even a clinically trivial gap reach significance.
Cohen's h (computed above) answers "how big," independent of sample size — a small
h alongside a tiny p-value would mean "real, but maybe not worth building the
system around," exactly the same caution Notebook 7's Cohen's d and Notebook 8's
Cramér's V bring to their respective test types.

### The multiple-comparisons trap — a caution for the whole system

If this notebook's approach (test one variable against the outcome) gets repeated
across every candidate input in the registry at $\alpha=0.05$ each, the chance of
*at least one* false "significant" finding by chance climbs fast.

In [ ]:
n_candidate_inputs = len(["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "slope", "ca", "thal"])
chance_of_false_positive = 1 - (1 - alpha) ** n_candidate_inputs
print(f"Testing all {n_candidate_inputs} candidate inputs at alpha=0.05 each:")
print(f"P(at least one false 'significant' finding by chance) = {chance_of_false_positive:.1%}")

corrected_alpha = alpha / n_candidate_inputs
print(f"\nBonferroni-corrected threshold per test: {corrected_alpha:.4f}")
print("Only findings beating this stricter bar should be trusted as individually significant")
print("when many inputs are screened this way at once.")

## Systems view — what this stage hands to the next one

This notebook validated *one* relationship (sex and disease rate) and, more
importantly, built the general framework and test-selection logic the next two
notebooks specialize. Notebook 7 applies the same logic to comparing continuous
measurements between groups; Notebook 8 applies it to comparing whole categorical
distributions.

## Try it yourself

1. Run the same proportion test comparing disease rate between `fbs==1` and
   `fbs==0` groups — is the gap statistically significant? How does its effect
   size compare to the sex-based gap above?
2. Compute the sample size that *would* be needed to reach 80% power for the
   observed sex-based effect size, using `NormalIndPower().solve_power()` — is this
   registry's actual size above or below that requirement?
3. Apply the Bonferroni-corrected threshold from the multiple-comparisons section
   to re-evaluate whether the sex/disease-rate finding still holds up as
   significant under the stricter bar.